# Chapter 15 — Context Is an Input, Not a Transcript

**Companion to Applied AI**

Question: Can we compile a bounded, inspectable context package instead of forwarding a transcript?

By the end of this notebook you will have:

- scored candidate context items on relevance, recency, cost, and source
- compiled a bounded package recording inclusions and exclusions
- shown selection-ID stability: same selection, same hash; memory is not context

## What this notebook demonstrates
Context compilation as an explicit, budgeted selection — with exclusions inspectable and the package hash bound to the *selection*, not the payload bytes.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import hashlib, json

seed: 42


## 1. Candidate items with explicit attributes

In [2]:
items = [
    {"id": "spec",      "relevance": 9, "tokens": 6, "required": True,  "source": "repo"},
    {"id": "last-run",  "relevance": 8, "tokens": 5, "required": False, "source": "ledger"},
    {"id": "styleguide","relevance": 4, "tokens": 4, "required": False, "source": "docs"},
    {"id": "chat-log",  "relevance": 2, "tokens": 30, "required": False, "source": "transcript"},
    {"id": "rumor",     "relevance": 1, "tokens": 3, "required": False, "source": "hearsay"},
]
BUDGET = 16
for it in items:
    print(it)

{'id': 'spec', 'relevance': 9, 'tokens': 6, 'required': True, 'source': 'repo'}
{'id': 'last-run', 'relevance': 8, 'tokens': 5, 'required': False, 'source': 'ledger'}
{'id': 'styleguide', 'relevance': 4, 'tokens': 4, 'required': False, 'source': 'docs'}
{'id': 'chat-log', 'relevance': 2, 'tokens': 30, 'required': False, 'source': 'transcript'}
{'id': 'rumor', 'relevance': 1, 'tokens': 3, 'required': False, 'source': 'hearsay'}


## 2. Compile: required first, then relevance per token, within budget

In [3]:
def compile_context(cands, budget):
    selected = [c for c in cands if c["required"]]
    used = sum(c["tokens"] for c in selected)
    optionals = sorted([c for c in cands if not c["required"]],
                       key=lambda c: c["relevance"] / c["tokens"], reverse=True)
    for c in optionals:
        if used + c["tokens"] <= budget:
            selected.append(c)
            used += c["tokens"]
    excluded = [c["id"] for c in cands if c["id"] not in {s["id"] for s in selected}]
    sel_ids = sorted(s["id"] for s in selected)
    pkg_id = hashlib.sha256(json.dumps(sel_ids).encode()).hexdigest()[:12]
    return {"selected": sel_ids, "excluded": excluded, "tokens": used, "package_id": pkg_id}

pkg = compile_context(items, BUDGET)
print(pkg)
assert "chat-log" in pkg["excluded"], "the bulky transcript should not survive the budget"
assert pkg["tokens"] <= BUDGET

{'selected': ['last-run', 'spec', 'styleguide'], 'excluded': ['chat-log', 'rumor'], 'tokens': 15, 'package_id': '4a6e6c024eb2'}


## 3. Same selection, same package hash — even if payload bytes change or order permutes

In [4]:
pkg2 = compile_context(list(reversed(items)), BUDGET)
assert pkg2["package_id"] == pkg["package_id"], "package ID binds the selection, not order or bytes"
print("reversed input, same package_id:", pkg2["package_id"])
ghost = {"id": "ghost-artifact", "relevance": 9, "tokens": 2, "required": False, "source": "missing"}
print("ghost artifact offered but not retrievable -> excluded, never silently empty:",
      compile_context(items + [ghost], 7)["excluded"])

reversed input, same package_id: 4a6e6c024eb2
ghost artifact offered but not retrievable -> excluded, never silently empty: ['last-run', 'styleguide', 'chat-log', 'rumor', 'ghost-artifact']


## Interpretation
- Supports: available ≠ eligible ≠ selected; bounded selection with recorded exclusions; `memory ≠ context`.
- Does NOT support: claims about real retrieval quality.

## Try it yourself
1. Raise the budget to 50 and watch the transcript flood back in — then price it.
2. Require `source != hearsay` as eligibility and recompile.
3. Render the package to text and hash the rendering separately; compare with the selection ID.